In [0]:
from pyspark.sql.functions import first, rand, count
dates = spark.sql("SELECT explode(sequence(DATE'2024-01-01', DATE'2024-03-24', INTERVAL 1 DAY)) as  calendar_date")
c_id = spark.sql("SELECT explode(sequence(1,200, 1)) as  client_id")
types = spark.sql("""SELECT concat("col_", colName) as col_name from (SELECT explode(sequence(1,20, 1)) as  colName)""")
 
dates = dates.repartition(99)
c_id = c_id.repartition(11)
types = types.repartition(1)
 
df_cartesian = c_id.crossJoin(dates.select("calendar_date")).crossJoin(types.select("col_name")).select("client_id","calendar_date","col_name")
df_cartesian2 = df_cartesian.groupBy("calendar_date").agg(count("client_id"))
 
# display(df_cartesian2.limit(1000))
 
df_cartesian = df_cartesian.withColumn("val", (rand()*10).cast("int"))

df_grp = df_cartesian.groupBy("client_id","calendar_date").pivot("col_name").agg((first("val").alias("val")))

display(df_grp)


client_id,calendar_date,col_1,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_2,col_20,col_3,col_4,col_5,col_6,col_7,col_8,col_9
170,2024-03-19,6,1,6,9,9,0,6,0,7,2,8,3,2,7,4,5,0,2,6,2
170,2024-03-07,5,1,2,7,1,7,9,0,3,6,0,5,3,4,9,4,6,7,0,9
170,2024-01-17,8,6,9,0,9,6,2,2,2,4,4,8,6,8,6,0,9,9,8,7
170,2024-01-31,4,0,7,3,8,5,0,3,5,9,6,1,4,5,7,2,0,6,7,4
170,2024-02-26,2,4,4,2,1,9,2,3,4,6,7,4,0,6,5,3,9,1,4,6
170,2024-03-21,9,8,0,6,8,8,9,3,6,5,2,1,6,0,6,6,5,6,5,4
170,2024-01-29,1,5,6,4,9,7,8,2,5,9,5,3,3,4,6,7,7,9,5,9
170,2024-03-24,4,7,0,2,7,1,8,3,6,2,0,4,9,2,3,8,3,5,3,2
170,2024-02-24,5,4,2,9,7,5,7,8,8,4,1,9,5,6,5,8,6,7,3,1
170,2024-02-29,9,6,3,8,8,7,5,9,1,9,0,3,5,6,9,9,6,0,8,7


# Create Dataframes

In [0]:
from pyspark.sql.functions import floor

df_1 = spark.sql("SELECT explode(sequence(1,900000, 1)) as  id_df_1")
duplicates = spark.sql("SELECT explode(sequence(1,100000, 1)) as  id_df_1")

df_1 = df_1.union(duplicates)

# Add dummy column
df_1 = df_1.withColumn("dummy_df_1", floor(rand() * 100))

Out[8]: 1000000

In [0]:
df_2 = spark.sql("SELECT explode(sequence(1,1000000, 1)) as  id_df_2")

# Add dummy column
df_2 = df_2.withColumn("dummy_df_2", floor(rand() * 100))

In [0]:
df_1.show(10)
df_2.show(10)

+-------+----------+
|id_df_1|dummy_df_1|
+-------+----------+
|      1|        39|
|      2|        95|
|      3|        64|
|      4|        69|
|      5|        81|
|      6|        56|
|      7|        77|
|      8|        87|
|      9|        44|
|     10|        79|
+-------+----------+
only showing top 10 rows

+-------+----------+
|id_df_2|dummy_df_2|
+-------+----------+
|      1|        10|
|      2|        99|
|      3|        31|
|      4|        58|
|      5|        62|
|      6|        85|
|      7|        49|
|      8|        33|
|      9|        83|
|     10|        69|
+-------+----------+
only showing top 10 rows



In [0]:
from pyspark.sql.functions import col

joinExpression = col("id_df_1") == col("id_df_2")

inner_joined = df_1.join(df_2, joinExpression, "inner")

df_1.join(df_2, joinExpression, "inner").explain()

inner_joined.show(10)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id_df_1#1669], [id_df_2#1729], Inner, BuildRight, false, true
   :- Project [id_df_1#1669, FLOOR((rand(3371024876895040115) * 100.0)) AS dummy_df_1#1675L]
   :  +- Union
   :     :- Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@ba20897f), false, [id_df_1#1669]
   :     :  +- Scan OneRowRelation[]
   :     +- Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@dd0b2c93), false, [id_df_1#1672]
   :        +- Scan OneRowRelation[]
   +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=5909]
      +- Project [id_df_2#1729, FLOOR((rand(-527247949767280938) * 100.0)) AS dummy_df_2#1731L]
         +- Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@d8890950), false, [id_df_2#1729]
            +- Scan OneRowRelation[]


+-------+----------+-------+----------+
|id_df_1|dummy_df_1|id_df_2|dummy_df_2|
+-------+----------+-------+--------

In [0]:
from pyspark.sql.functions import col

joinExpression = col("id_df_1") == col("id_df_2")

left_joined = df_1.join(df_2, joinExpression, "left")

df_1.join(df_2, joinExpression, "left").explain()

left_joined.show(10)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id_df_1#1669], [id_df_2#1729], LeftOuter, BuildRight, false, true
   :- Project [id_df_1#1669, FLOOR((rand(3371024876895040115) * 100.0)) AS dummy_df_1#1675L]
   :  +- Union
   :     :- Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@ba20897f), false, [id_df_1#1669]
   :     :  +- Scan OneRowRelation[]
   :     +- Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@dd0b2c93), false, [id_df_1#1672]
   :        +- Scan OneRowRelation[]
   +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=6197]
      +- Project [id_df_2#1729, FLOOR((rand(-527247949767280938) * 100.0)) AS dummy_df_2#1731L]
         +- Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@d8890950), false, [id_df_2#1729]
            +- Scan OneRowRelation[]


+-------+----------+-------+----------+
|id_df_1|dummy_df_1|id_df_2|dummy_df_2|
+-------+----------+-------+----

# Drop Duplicates

In [0]:
left_joined = left_joined.drop(col("id_df_1")).select("dummy_df_2")

left_joined.show(10)

+----------+
|dummy_df_2|
+----------+
|        10|
|        99|
|        31|
|        58|
|        62|
|        85|
|        49|
|        33|
|        83|
|        69|
+----------+
only showing top 10 rows

